# 🎰 Malibu: Casino Floor & Cage Management System
## Interactive JDBC Architecture, DBMS Theory & Implementation Guide

This Jupyter Notebook serves as the comprehensive, interactive companion to the **Malibu Casino Floor & Cage Management System** and its Java CLI application (`CasinoApp.java`).

### 📌 Objectives of this Notebook:
1. **Explain the End-to-End JDBC Architecture**: How Java applications establish, configure, and manage connections to a relational database engine (MySQL).
2. **Detailed DBMS Concept Mapping**: For every single menu option in `CasinoApp.java`, explain the underlying Database Management System concepts (Relational Joins, Constraints, DML, Stored Procedures, Triggers, Transactions & ACID properties, and Analytical Aggregation).
3. **Interactive Python-to-JDBC Demonstration**: Use `JayDeBeApi` to load the exact same JDBC Driver (`mysql-connector-j-26.10.0.jar`) and execute each query live via the Java Virtual Machine.

---
### 🏛️ How JDBC Connects with MySQL: Architecture & Lifecycle

Java Database Connectivity (**JDBC**) is a vendor-independent Java API for executing SQL statements and managing relational database sessions.

```
+-------------------------------------------------------------------------+
|                  Java Application Layer (CasinoApp.java)                |
+-------------------------------------------------------------------------+
                                    |
                                    v
+-------------------------------------------------------------------------+
|                         JDBC API (java.sql.*)                           |
|      DriverManager  |  Connection  |  PreparedStatement  |  ResultSet   |
+-------------------------------------------------------------------------+
                                    |
                                    v
+-------------------------------------------------------------------------+
|             MySQL Connector/J Driver (mysql-connector-j-*.jar)          |
|                   Class: com.mysql.cj.jdbc.Driver                       |
+-------------------------------------------------------------------------+
                                    |
                          TCP/IP Socket (Port 3306)
                                    |
                                    v
+-------------------------------------------------------------------------+
|                  MySQL Database Engine 8.0 ('malibu')                   |
|     InnoDB Engine  |  Tables  |  Triggers  |  Stored Procedures         |
+-------------------------------------------------------------------------+
```

#### The 5 Fundamental Steps of JDBC:
1. **Driver Loading & Registration**: The driver class `com.mysql.cj.jdbc.Driver` is loaded into memory and registered with `DriverManager`.
2. **Connection Establishment**: `DriverManager.getConnection(url, user, password)` opens a TCP/IP socket connection to MySQL on port 3306 and negotiates authentication.
3. **Statement Creation**:
   - `PreparedStatement`: Pre-compiled SQL with `?` parameter placeholders (prevents SQL injection).
   - `CallableStatement`: Invokes pre-compiled Stored Procedures and User-Defined Functions with `IN` and `OUT` parameters.
4. **Execution & Cursor Handling**:
   - `executeQuery()` returns a `ResultSet` cursor for tabular data retrieval (`SELECT`).
   - `executeUpdate()` returns an integer representing affected rows (`INSERT`, `UPDATE`, `DELETE`).
5. **Resource Management**: All database resources (`Connection`, `Statement`, `ResultSet`) are closed deterministically using Java's `try-with-resources` construct to prevent memory and connection pool leaks.


---
### 🛠️ Step 1: Environment & Dependency Setup

We import `os`, `jaydebeapi`, and `pandas`.
- **`jaydebeapi`**: A Python module that connects to Java JDBC drivers via JPype (the Java Virtual Machine bridge). This ensures our Python notebook runs through the exact same MySQL Connector/J driver bytecode as `CasinoApp.java`.
- **`pandas`**: Formats tabular `ResultSet` outputs into clean visual representations.


In [1]:
import os
import jaydebeapi
import pandas as pd

print("✅ Python JDBC Environment ready.")
print(f"Working Directory: {os.getcwd()}")


✅ Python JDBC Environment ready.
Working Directory: D:\Malibu


---
### 🔑 Step 2: External Credential Ingestion (`db.properties`)

#### 🧠 DBMS & Software Engineering Concept:
- **Defense in Depth & 12-Factor App Configuration**: Sensitive database credentials (passwords, usernames, hostnames) must **never** be hardcoded inside source files.
- In `CasinoApp.java`, credentials are read from `src/db.properties` using `java.io.FileInputStream` and `java.util.Properties`.
- This ensures environmental portability (development, testing, production) without recompilation.


In [2]:
def load_db_properties(prop_path="src/db.properties"):
    if not os.path.exists(prop_path):
        prop_path = "db.properties"
    if not os.path.exists(prop_path):
        raise FileNotFoundError(f"Configuration file not found at {prop_path}")
        
    config = {}
    with open(prop_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                config[key.strip()] = val.strip()
    return config

# Load credentials
db_config = load_db_properties()
db_url = db_config.get("db.url")
db_user = db_config.get("db.user")
db_password = db_config.get("db.password")

print("✅ Configuration loaded successfully from db.properties:")
print(f"  URL  : {db_url}")
print(f"  User : {db_user}")
print(f"  Pass : {'*' * len(db_password)} (Masked for security)")


✅ Configuration loaded successfully from db.properties:
  URL  : jdbc:mysql://localhost:3306/malibu?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC
  User : root
  Pass : **************** (Masked for security)


---
### 🔌 Step 3: Establishing the Live JDBC Connection

#### 🧠 DBMS Concepts:
- **Database Session Handshake**: Establishing a TCP/IP socket connection to MySQL 8.0 on port 3306.
- **Driver Specification**: `com.mysql.cj.jdbc.Driver` located in `lib/mysql-connector-j-26.10.0.jar`.
- **Session Verification**: We query MySQL system metadata `SELECT DATABASE(), VERSION(), CURRENT_TIMESTAMP()` to confirm the active schema and engine status.


In [3]:
jar_path = os.path.abspath("lib/mysql-connector-j-26.10.0.jar")
driver_class = "com.mysql.cj.jdbc.Driver"

print(f"Driver JAR Path: {jar_path}")
assert os.path.exists(jar_path), f"JAR file missing at {jar_path}"

# Connect via JayDeBeApi (Java Virtual Machine JDBC bridge)
conn = jaydebeapi.connect(
    driver_class,
    db_url,
    [db_user, db_password],
    jar_path
)

# Verify connection
cursor = conn.cursor()
cursor.execute("SELECT DATABASE(), VERSION(), CURRENT_TIMESTAMP()")
row = cursor.fetchone()
print("\n🎉 JDBC Connection Established Successfully!")
print(f"  Active Database : {row[0]}")
print(f"  MySQL Version   : {row[1]}")
print(f"  Timestamp       : {row[2]}")
cursor.close()


Driver JAR Path: D:\Malibu\lib\mysql-connector-j-26.10.0.jar



🎉 JDBC Connection Established Successfully!
  Active Database : malibu
  MySQL Version   : 8.0.46
  Timestamp       : 2026-10-29 06:41:02


---
### 📋 CLI Command 1: [SELECT] View All Registered Players

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, the method `viewAllPlayers()` executes:
```java
String sql = "SELECT player_id, full_name, govt_id, age, chip_balance, comp_points, tier_name " +
             "FROM Players NATURAL JOIN VIP_Tiers ORDER BY player_id";
PreparedStatement pstmt = con.prepareStatement(sql);
ResultSet rs = pstmt.executeQuery();
```

#### 🧠 DBMS Concepts Used:
1. **Relational Join (`NATURAL JOIN`)**:
   - Joins table `Players` with `VIP_Tiers` based on their common attribute (`tier_id`).
   - Eliminates redundant foreign key columns from the Cartesian product:
     $$\Pi_{\text{player\_id, full\_name, govt\_id, age, chip\_balance, comp\_points, tier\_name}}(\text{Players} \bowtie \text{VIP\_Tiers})$$
2. **Projection & Selection**: Only relevant attributes are retrieved from disk, minimizing network transfer overhead.
3. **Ordering**: `ORDER BY player_id` performs server-side B-Tree index traversal on the Primary Key.
4. **Cursor-Based Retrieval**: JDBC `ResultSet` behaves as a client-side cursor navigating one record at a time via `rs.next()`.


In [4]:
def view_all_players(connection):
    sql = """
        SELECT player_id, full_name, govt_id, age, chip_balance, comp_points, tier_name
        FROM Players NATURAL JOIN VIP_Tiers
        ORDER BY player_id
    """
    curs = connection.cursor()
    curs.execute(sql)
    rows = curs.fetchall()
    cols = ["Player ID", "Full Name", "Govt ID", "Age", "Chip Balance", "Comp Points", "VIP Tier"]
    
    clean_rows = []
    for r in rows:
        clean_rows.append([
            int(str(r[0])),
            str(r[1]),
            str(r[2]),
            int(str(r[3])),
            float(str(r[4])),
            int(str(r[5])),
            str(r[6])
        ])
    curs.close()
    return pd.DataFrame(clean_rows, columns=cols)

df_players = view_all_players(conn)
print(f"Total Players Found: {len(df_players)}")
df_players.head(10)


Total Players Found: 14


,Player ID,Full Name,Govt ID,Age,Chip Balance,Comp Points,VIP Tier
0,101,Aarav Mehta,PAN-A1098X,26,206000.0,450,Gold
1,102,Divya Reddy,PAN-B2234Y,29,160000.0,820,Platinum
2,103,Karan Malhotra,PAN-C7761Z,34,620000.0,2400,High Roller
3,104,Sneha Mukherjee,PAN-D9012K,24,25000.0,120,Bronze
4,105,Aditya Singhania,PAN-E3345M,41,1250000.0,7800,Royal Whale
5,106,Pooja Hegde,PAN-F5567P,28,80000.0,310,Silver
6,107,Varun Dhawan,PAN-G8890Q,31,210000.0,1150,Diamond
7,108,Ishaan Khattar,PAN-H1243R,23,15000.0,80,Bronze
8,109,Rhea Chakraborty,PAN-J5678S,30,430000.0,1950,Sapphire
9,110,Sameer Khan,PAN-K9087T,38,310000.0,1400,Ruby


---
### ➕ CLI Command 2: [INSERT] Register a New Player

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `registerPlayer(Scanner scanner)` executes:
```java
String sql = "INSERT INTO Players (full_name, govt_id, age, chip_balance, comp_points, tier_id) VALUES (?, ?, ?, ?, 0, ?)";
PreparedStatement pstmt = con.prepareStatement(sql, Statement.RETURN_GENERATED_KEYS);
pstmt.executeUpdate();
ResultSet keys = pstmt.getGeneratedKeys();
```

#### 🧠 DBMS Concepts Used:
1. **Primary Key Auto-Incrementation**:
   - `player_id` is defined as `INT PRIMARY KEY AUTO_INCREMENT`.
   - The database engine atomically generates sequential, non-colliding surrogate keys.
2. **Domain Integrity & Check Constraints**:
   - `CHECK (age >= 21)`: Enforced at the schema level. If age < 21 is submitted, MySQL rejects the insert with constraint violation error code `3819`.
   - `UNIQUE (govt_id)`: Enforces Know Your Customer (KYC) identity uniqueness across patrons.
3. **Referential Integrity**:
   - `FOREIGN KEY (tier_id) REFERENCES VIP_Tiers(tier_id)`: Guarantees every player belongs to an existing loyalty tier.
4. **SQL Injection Immunity**:
   - Using parameterized placeholders (`?`) ensures user input is strictly treated as literal data rather than executable SQL syntax.


In [5]:
import random

def register_player(connection, name, govt_id, age, balance, tier_id):
    sql = """
        INSERT INTO Players (full_name, govt_id, age, chip_balance, comp_points, tier_id)
        VALUES (?, ?, ?, ?, 0, ?)
    """
    curs = connection.cursor()
    try:
        curs.execute(sql, (name, govt_id, int(str(age)), float(str(balance)), int(str(tier_id))))
        curs.execute("SELECT LAST_INSERT_ID()")
        new_id = int(str(curs.fetchone()[0]))
        curs.close()
        return new_id
    except Exception as e:
        curs.close()
        raise e

# Register a demo player with unique KYC PAN
rand_num = random.randint(1000, 9999)
demo_govt_id = f"PAN-DEMO{rand_num}"
new_player_id = register_player(conn, "Vikramaditya Roy", demo_govt_id, 28, 137000.0, 3)

print(f"✅ [INSERT SUCCESS] Player Registered!")
print(f"  Assigned Player ID : {new_player_id}")
print(f"  KYC Govt ID        : {demo_govt_id}")
print(f"  Tier ID            : 3 (Gold Tier)")


✅ [INSERT SUCCESS] Player Registered!
  Assigned Player ID : 115
  KYC Govt ID        : PAN-DEMO8649
  Tier ID            : 3 (Gold Tier)


---
### 🔄 CLI Command 3: [UPDATE] Modify Player Chip Balance

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `updatePlayerBalance(Scanner scanner)` executes:
```java
String sql = "UPDATE Players SET chip_balance = ? WHERE player_id = ?";
PreparedStatement pstmt = con.prepareStatement(sql);
pstmt.setBigDecimal(1, newBalance);
pstmt.setInt(2, playerId);
int affectedRows = pstmt.executeUpdate();
```

#### 🧠 DBMS Concepts Used:
1. **Data Manipulation (DML - UPDATE)**:
   - Modifies existing row values in-place under an index-assisted lock on `player_id`.
2. **Integrity Enforcement**:
   - `CHECK (chip_balance >= 0)`: The DBMS guarantees that balances cannot be set to negative values.
3. **Execution Feedback & Concurrency**:
   - `executeUpdate()` returns the exact row count modified (`affectedRows`). If `affectedRows == 0`, the application detects that the specified `player_id` does not exist.


In [6]:
def update_player_balance(connection, player_id, new_balance):
    sql = "UPDATE Players SET chip_balance = ? WHERE player_id = ?"
    curs = connection.cursor()
    curs.execute(sql, (float(str(new_balance)), int(str(player_id))))
    # Fetch updated balance for verification
    curs.execute("SELECT player_id, full_name, chip_balance FROM Players WHERE player_id = ?", (int(str(player_id)),))
    row = curs.fetchone()
    curs.close()
    return (int(str(row[0])), str(row[1]), float(str(row[2])))

updated_record = update_player_balance(conn, new_player_id, 150000.00)
print(f"✅ [UPDATE SUCCESS] Player ID {updated_record[0]} ({updated_record[1]}):")
print(f"  Updated Chip Balance : INR {updated_record[2]:,.2f}")


✅ [UPDATE SUCCESS] Player ID 115 (Vikramaditya Roy):
  Updated Chip Balance : INR 150,000.00


---
### 🗑️ CLI Command 4: [DELETE] Decommission a Game Table

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `deleteGameTable(Scanner scanner)` executes:
```java
String sql = "DELETE FROM Game_Tables WHERE table_code = ?";
PreparedStatement pstmt = con.prepareStatement(sql);
pstmt.setString(1, tableCode);
int rows = pstmt.executeUpdate();
```

#### 🧠 DBMS Concepts Used:
1. **Relational Deletion (DML - DELETE)**:
   - Permanently deletes records satisfying the search condition (`WHERE table_code = ?`).
2. **Referential Integrity & Foreign Key Enforcement**:
   - The table `Game_Tables` is referenced by `Betting_Sessions(table_id)`.
   - If active gaming sessions reference a table, MySQL halts deletion and raises error code `23000` (Foreign Key Constraint Violation), preventing orphaned records.
3. **Row Affected Validation**:
   - Differentiates between a successful table decommission (`rows > 0`) and a missing table code (`rows == 0`).


In [7]:
def decommission_table(connection, table_code):
    # Ensure temporary demo table exists first
    curs = connection.cursor()
    curs.execute("""
        INSERT INTO Game_Tables (table_code, game_type, min_bet, max_bet, table_chip_float, dealer_id)
        SELECT ?, 'Blackjack', 5000.0, 100000.0, 1000000.0, NULL
        WHERE NOT EXISTS (SELECT 1 FROM Game_Tables WHERE table_code = ?)
    """, (table_code, table_code))
    
    # Now execute DELETE
    delete_sql = "DELETE FROM Game_Tables WHERE table_code = ?"
    curs.execute(delete_sql, (table_code,))
    curs.close()
    return True

demo_table_code = "TEMP-99"
decommission_table(conn, demo_table_code)
print(f"✅ [DELETE SUCCESS] Decommissioned and removed table '{demo_table_code}' from active inventory.")


✅ [DELETE SUCCESS] Decommissioned and removed table 'TEMP-99' from active inventory.


---
### ⚙️ CLI Command 5: [CALLABLE] Process Chip Buy-In (Stored Procedure)

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `processChipBuyIn(Scanner scanner)` executes:
```java
String sql = "{CALL ProcessChipBuyIn(?, ?, ?, ?, ?)}";
CallableStatement cstmt = con.prepareCall(sql);
cstmt.setInt(1, playerId);
cstmt.setInt(2, cashierId);
cstmt.setBigDecimal(3, amount);
cstmt.setBoolean(4, giveBonus);
cstmt.registerOutParameter(5, Types.VARCHAR);
cstmt.execute();
String status = cstmt.getString(5);
```

#### 🧠 DBMS Concepts Used:
1. **Server-Side Stored Procedures (`CREATE PROCEDURE`)**:
   - Business logic is pre-compiled and executed directly on the MySQL server, reducing network round-trips.
2. **Procedural Transaction & Savepoints**:
   - Inside `ProcessChipBuyIn`:
     ```sql
     START TRANSACTION;
     INSERT INTO Cage_Transactions ... ;
     UPDATE Players SET chip_balance = chip_balance + p_amount ... ;
     SAVEPOINT purchased_chips_credited;
     IF p_give_bonus = TRUE THEN
         IF p_amount < 50000.00 THEN
             ROLLBACK TO SAVEPOINT purchased_chips_credited; -- Revert bonus only
         ELSE
             UPDATE Players SET chip_balance = chip_balance + 2000.00;
         END IF;
     END IF;
     COMMIT;
     ```
3. **Exception Handlers**:
   - `DECLARE EXIT HANDLER FOR SQLEXCEPTION` automatically rolls back the entire transaction if any constraint is violated.
4. **CallableStatement & OUT Parameters**:
   - Java uses `registerOutParameter(5, Types.VARCHAR)` to retrieve the procedure's output status message.


In [8]:
def process_chip_buyin(connection, player_id, cashier_id, amount, give_bonus):
    curs = connection.cursor()
    call_sql = """
        CALL ProcessChipBuyIn(?, ?, ?, ?, @out_msg);
    """
    curs.execute(call_sql, (int(str(player_id)), int(str(cashier_id)), float(str(amount)), bool(give_bonus)))
    curs.execute("SELECT @out_msg")
    msg = str(curs.fetchone()[0])
    curs.close()
    return msg

status_msg = process_chip_buyin(conn, 101, 11, 75000.00, True)
print(f"✅ [STORED PROCEDURE RESULT]:")
print(f"  Target Player ID : 101")
print(f"  Amount           : INR 75,000.00 (Eligible for bonus)")
print(f"  Procedure Status : {status_msg}")


✅ [STORED PROCEDURE RESULT]:
  Target Player ID : 101
  Amount           : INR 75,000.00 (Eligible for bonus)
  Procedure Status : SUCCESS: Chips credited with 2,000 promotional bonus!


---
### 🚨 CLI Command 6: [TRIGGER] Cash-Out System with AML Trigger Alert

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `processCashOut(Scanner scanner)` implements the requirement from `projectDetail.txt`:
```java
// Insert with is_flagged = FALSE (client default)
String insertSql = "INSERT INTO Cage_Transactions (player_id, cashier_id, txn_type, amount, is_flagged) VALUES (?, ?, 'CASH_OUT', ?, FALSE)";
// Database trigger evaluates amount and sets is_flagged = TRUE if amount >= 500,000
// Java verifies is_flagged from database:
if (isFlagged) {
    System.out.println("*** ALERT NOTIFICATION ***");
    System.out.println("SUSPICIOUS HIGH VALUE CASH-OUT FLAGGED BY AML TRIGGER!");
    System.out.println("NOTIFICATION: TRANSACTION WILL NOT BE DONE.");
    // Payout blocked: Chip balance is NOT deducted!
} else {
    // Normal transaction: Deduct chip balance
    UPDATE Players SET chip_balance = chip_balance - ? WHERE player_id = ?;
}
```

#### 🧠 DBMS Concepts Used:
1. **Active Databases & Triggers**:
   - Event-Condition-Action (ECA) rule paradigm.
   - Timing: `BEFORE INSERT ON Cage_Transactions FOR EACH ROW`.
2. **Trigger Logic (`FlagForHighCashOut`)**:
   ```sql
   CREATE TRIGGER FlagForHighCashOut
   BEFORE INSERT ON Cage_Transactions
   FOR EACH ROW
   BEGIN
       IF NEW.txn_type = 'CASH_OUT' AND NEW.amount >= 500000.00 THEN
           SET NEW.is_flagged = TRUE;
       END IF;
   END;
   ```
3. **Automated AML (Anti-Money Laundering) Compliance**:
   - The trigger automatically overrides client parameters and flags transactions $\ge$ 500,000 INR.
4. **Conditional Business Workflow**:
   - If the database writes `is_flagged = 1`, the Java application sends an **Alert Notification** and aborts the redemption, safeguarding casino liquidity.


In [9]:
def test_cashout_system(connection, player_id, cashier_id, amount):
    curs = connection.cursor()
    # 1. Fetch current balance
    curs.execute("SELECT chip_balance FROM Players WHERE player_id = ?", (int(str(player_id)),))
    current_balance = float(str(curs.fetchone()[0]))
    amount = float(str(amount))
    
    if current_balance < amount:
        curs.close()
        return {"status": "INSUFFICIENT_FUNDS", "balance": current_balance}
        
    # 2. Insert with client parameter is_flagged = FALSE
    insert_sql = """
        INSERT INTO Cage_Transactions (player_id, cashier_id, txn_type, amount, is_flagged)
        VALUES (?, ?, 'CASH_OUT', ?, FALSE)
    """
    curs.execute(insert_sql, (int(str(player_id)), int(str(cashier_id)), amount))
    curs.execute("SELECT LAST_INSERT_ID()")
    txn_id = int(str(curs.fetchone()[0]))
    
    # 3. Query back the trigger-modified flag
    curs.execute("SELECT is_flagged FROM Cage_Transactions WHERE txn_id = ?", (int(str(txn_id)),))
    is_flagged = bool(curs.fetchone()[0])
    
    if is_flagged:
        # Payout BLOCKED!
        curs.close()
        return {
            "status": "BLOCKED_AML_ALERT",
            "txn_id": txn_id,
            "amount": amount,
            "is_flagged": is_flagged,
            "notification": "TRANSACTION WILL NOT BE DONE. Flagged by AML audit trigger!"
        }
    else:
        # Payout APPROVED: Deduct balance
        curs.execute("UPDATE Players SET chip_balance = chip_balance - ? WHERE player_id = ?", (amount, int(str(player_id))))
        curs.close()
        return {
            "status": "APPROVED",
            "txn_id": txn_id,
            "amount": amount,
            "is_flagged": is_flagged,
            "remaining_balance": current_balance - amount
        }

# Scenario A: High-value cash-out (trigger fires)
res_flagged = test_cashout_system(conn, 105, 12, 600000.00)
print("=" * 60)
print("TEST CASE 1: High Value Cash-Out (INR 600,000.00)")
print("=" * 60)
print(f"  Status        : {res_flagged['status']}")
print(f"  Txn ID        : {res_flagged['txn_id']}")
print(f"  Database Flag : is_flagged = {res_flagged['is_flagged']}")
print(f"  ALERT NOTICE  : {res_flagged['notification']}")

# Scenario B: Standard cash-out under threshold
res_normal = test_cashout_system(conn, 101, 11, 1000.00)
print("\n" + "=" * 60)
print("TEST CASE 2: Standard Cash-Out (INR 1,000.00)")
print("=" * 60)
print(f"  Status        : {res_normal['status']}")
print(f"  Txn ID        : {res_normal['txn_id']}")
print(f"  Database Flag : is_flagged = {res_normal['is_flagged']}")
print(f"  Payout        : Approved! Remaining chips: INR {res_normal['remaining_balance']:,.2f}")


TEST CASE 1: High Value Cash-Out (INR 600,000.00)
  Status        : BLOCKED_AML_ALERT
  Txn ID        : 925
  Database Flag : is_flagged = True
  ALERT NOTICE  : TRANSACTION WILL NOT BE DONE. Flagged by AML audit trigger!

TEST CASE 2: Standard Cash-Out (INR 1,000.00)
  Status        : APPROVED
  Txn ID        : 926
  Database Flag : is_flagged = False
  Payout        : Approved! Remaining chips: INR 282,000.00


---
### 🛡️ CLI Command 7: [TRANSACTION] Peer-to-Peer Transfer (ACID Control)

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `transferChipsTransaction(Scanner scanner)` implements explicit JDBC transaction control:
```java
con.setAutoCommit(false); // Begin explicit transaction boundary
try {
    // Step 1: Deduct from sender
    deductStmt.executeUpdate();
    // Step 2: Mid-point failure check
    if (simulateError) throw new RuntimeException("Crash simulation!");
    // Step 3: Credit to receiver
    creditStmt.executeUpdate();
    // Step 4: Commit transaction
    con.commit();
} catch (Exception e) {
    con.rollback(); // Rollback on failure
} finally {
    con.setAutoCommit(true);
}
```

#### 🧠 DBMS Concepts Used:
1. **ACID Properties**:
   - **Atomicity**: The deduction and credit form a single, indivisible logical unit of work. Both happen, or neither happens.
   - **Consistency**: Casino total chip float is conserved. Zero funds disappear or appear out of nowhere.
   - **Isolation**: Intermediate states (funds deducted from sender but not yet credited to receiver) are hidden from concurrent transactions.
   - **Durability**: Once `con.commit()` executes, the changes survive system failures and power outages.
2. **Transaction Boundaries**:
   - Turning off auto-commit (`con.setAutoCommit(false)`) stops MySQL from committing each SQL statement individually.
   - `con.commit()` flushes the Write-Ahead Log (WAL) and commits changes.
   - `con.rollback()` restores affected rows from the undo log.


In [10]:
def run_transaction_demo(connection, sender_id, receiver_id, amount, simulate_failure=False):
    curs = connection.cursor()
    # In JDBC: con.setAutoCommit(false)
    # In MySQL raw SQL: START TRANSACTION
    curs.execute("START TRANSACTION")
    
    amount_val = float(str(amount))
    s_id = int(str(sender_id))
    r_id = int(str(receiver_id))
    
    try:
        # Step 1: Deduct from sender
        curs.execute("UPDATE Players SET chip_balance = chip_balance - ? WHERE player_id = ? AND chip_balance >= ?",
                     (amount_val, s_id, amount_val))
        
        # Step 2: Mid-point crash simulation
        if simulate_failure:
            raise RuntimeError("💥 Simulated mid-transaction hardware crash / network drop!")
            
        # Step 3: Credit to receiver
        curs.execute("UPDATE Players SET chip_balance = chip_balance + ? WHERE player_id = ?",
                     (amount_val, r_id))
        
        # Step 4: Commit
        curs.execute("COMMIT")
        curs.close()
        return {"status": "COMMITTED", "msg": f"Transferred INR {amount_val:,.2f} from Player {s_id} to Player {r_id}"}
        
    except Exception as e:
        # Step 5: Rollback
        curs.execute("ROLLBACK")
        curs.close()
        return {"status": "ROLLED_BACK", "msg": str(e)}

# Demo A: Transaction with simulated failure -> Rolled Back!
res_aborted = run_transaction_demo(conn, 101, 102, 5000.0, simulate_failure=True)
print("❌ TRANSACTION SIMULATED FAILURE:")
print(f"  Status : {res_aborted['status']}")
print(f"  Reason : {res_aborted['msg']}")
print("  Action : con.rollback() invoked. Zero balance discrepancy, integrity preserved.")

# Demo B: Successful transaction -> Committed!
res_committed = run_transaction_demo(conn, 101, 102, 5000.0, simulate_failure=False)
print("\n✅ TRANSACTION SUCCESSFUL COMMIT:")
print(f"  Status : {res_committed['status']}")
print(f"  Result : {res_committed['msg']}")


❌ TRANSACTION SIMULATED FAILURE:
  Status : ROLLED_BACK
  Reason : 💥 Simulated mid-transaction hardware crash / network drop!
  Action : con.rollback() invoked. Zero balance discrepancy, integrity preserved.

✅ TRANSACTION SUCCESSFUL COMMIT:
  Status : COMMITTED
  Result : Transferred INR 5,000.00 from Player 101 to Player 102


---
### 📊 CLI Command 8: [ANALYTICAL] Floor Inventory Summary Report

#### 💻 Java CLI Implementation:
In `CasinoApp.java`, `viewFloorInventoryReport()` executes:
```java
String sql = "SELECT game_type, " +
             "       COUNT(table_id) AS total_tables, " +
             "       MIN(min_bet) AS lowest_min_bet, " +
             "       MAX(max_bet) AS highest_max_bet, " +
             "       SUM(table_chip_float) AS total_chip_inventory, " +
             "       AVG(table_chip_float) AS avg_table_float " +
             "FROM Game_Tables " +
             "GROUP BY game_type " +
             "ORDER BY total_chip_inventory DESC";
PreparedStatement pstmt = con.prepareStatement(sql);
ResultSet rs = pstmt.executeQuery();
```

#### 🧠 DBMS Concepts Used:
1. **Relational Aggregation**:
   - `COUNT(table_id)`: Multiplicity of physical tables deployed.
   - `MIN(min_bet)` & `MAX(max_bet)`: Dynamic betting range boundary per game type.
   - `SUM(table_chip_float)` & `AVG(table_chip_float)`: Total float and mean capitalization per game type.
2. **Grouping (`GROUP BY game_type`)**:
   - Partitions tuples into disjoint equivalence classes matching each unique `game_type` (Blackjack, Poker, Roulette, Baccarat).
3. **Sorting Aggregate Results**:
   - `ORDER BY total_chip_inventory DESC` orders the summary output by highest casino capital exposure.


In [11]:
def view_floor_inventory_report(connection):
    sql = """
        SELECT game_type,
               COUNT(table_id) AS total_tables,
               MIN(min_bet) AS lowest_min_bet,
               MAX(max_bet) AS highest_max_bet,
               SUM(table_chip_float) AS total_chip_inventory,
               AVG(table_chip_float) AS avg_table_float
        FROM Game_Tables
        GROUP BY game_type
        ORDER BY total_chip_inventory DESC
    """
    curs = connection.cursor()
    curs.execute(sql)
    rows = curs.fetchall()
    cols = ["Game Type", "Total Tables", "Lowest Min Bet (INR)", "Highest Max Bet (INR)", "Total Chip Float (INR)", "Average Float (INR)"]
    
    clean_rows = []
    for r in rows:
        clean_rows.append([
            str(r[0]),
            int(str(r[1])),
            float(str(r[2])) if r[2] is not None else 0.0,
            float(str(r[3])) if r[3] is not None else 0.0,
            float(str(r[4])) if r[4] is not None else 0.0,
            float(str(r[5])) if r[5] is not None else 0.0
        ])
    curs.close()
    return pd.DataFrame(clean_rows, columns=cols)

df_report = view_floor_inventory_report(conn)
df_report


,Game Type,Total Tables,Lowest Min Bet (INR),Highest Max Bet (INR),Total Chip Float (INR),Average Float (INR)
0,Blackjack,4,1000.0,100000.0,3650000.0,9.125000e+05
1,Poker,2,5000.0,250000.0,3200000.0,1.600000e+06
2,Baccarat,2,2000.0,100000.0,2400000.0,1.200000e+06
3,Roulette,3,500.0,50000.0,1850000.0,6.166667e+05


---
### 🎓 Summary & DBMS Viva Voce Reference Table

| CLI Menu Command | Java Method in `CasinoApp.java` | Primary SQL / Routine | Core DBMS Concept |
| :--- | :--- | :--- | :--- |
| **1. View All Players** | `viewAllPlayers()` | `SELECT ... FROM Players NATURAL JOIN VIP_Tiers` | Relational Join (`NATURAL JOIN`), Projection, B-Tree Index Scan |
| **2. Register Player** | `registerPlayer()` | `INSERT INTO Players (...) VALUES (?, ?, ...)` | Surrogate Key (`AUTO_INCREMENT`), `CHECK (age >= 21)`, `UNIQUE` KYC |
| **3. Update Balance** | `updatePlayerBalance()` | `UPDATE Players SET chip_balance = ? WHERE ...` | In-Place DML, Domain Integrity (`chip_balance >= 0`), Affected Rows Check |
| **4. Decommission Table** | `deleteGameTable()` | `DELETE FROM Game_Tables WHERE table_code = ?` | Relational Deletion, Referential Integrity (`FOREIGN KEY` cascade/restrict) |
| **5. Process Buy-In** | `processChipBuyIn()` | `{CALL ProcessChipBuyIn(?, ?, ?, ?, ?)}` | Stored Procedure, CallableStatement, `SAVEPOINT`, Exception Handler |
| **6. Cash-Out System** | `processCashOut()` | `FlagForHighCashOut` (Trigger on `Cage_Transactions`) | Active Databases, `BEFORE INSERT` Trigger, ECA Rule, AML Alert |
| **7. Transfer Chips** | `transferChipsTransaction()` | `con.setAutoCommit(false)`, `commit()`, `rollback()` | ACID Properties (Atomicity, Consistency, Isolation, Durability) |
| **8. Floor Report** | `viewFloorInventoryReport()` | `SELECT ... GROUP BY game_type ORDER BY ...` | Relational Aggregation (`COUNT`, `MIN`, `MAX`, `SUM`, `AVG`), Equivalence Grouping |

---
### 🛑 Cleanup: Closing Database Connections
Deterministic closure of database resources ensures no dangling locks or connection leaks.


In [12]:
# Close the JDBC connection
conn.close()
print("🔒 JDBC Connection successfully closed. Resources released.")


🔒 JDBC Connection successfully closed. Resources released.
